In [173]:
import json
from pathlib import Path
from typing import Any
import boto3
import pandas as pd
from dotenv import load_dotenv

# Load AWS credentials from .env
load_dotenv()

# ---- Paths ----
PROJECT_ROOT = Path("..").resolve()  # notebook is in /notebooks
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROMPT_DIR = PROJECT_ROOT / "prompts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRANSACTION_FILE = DATA_DIR / "transactions.csv"

# ---- AWS / Bedrock config ----
AWS_REGION = "us-east-1"   # change to us-west-2 IF your Bedrock console is in that region
BEDROCK_MODEL_ID = "amazon.nova-micro-v1:0"

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)

def load_prompt(filename: str) -> str:
    return (PROMPT_DIR / filename).read_text()


In [174]:
def invoke_bedrock_json(system_prompt: str, user_prompt: str, max_tokens: int = 1024) -> Any:
    """
    Invoke Amazon Nova Micro and return parsed JSON if possible.
    """

    request_body = {
        "schemaVersion": "messages-v1",
        "system": [{"text": system_prompt}],
        "messages": [
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": max_tokens,
            "temperature": 0.2,
            "topP": 0.9
        }
    }

    response = bedrock.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(request_body),
        contentType="application/json",
        accept="application/json"
    )

    body = json.loads(response["body"].read())
    text = body["output"]["message"]["content"][0]["text"]

    # Try to parse JSON
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        print("Model did NOT return valid JSON. Returning raw text.")
        return {"raw_text": text}


In [175]:
def invoke_bedrock_text(system_prompt: str, user_prompt: str, max_tokens: int = 512, temperature: float = 0.2) -> str:
    request_body = {
        "schemaVersion": "messages-v1",
        "system": [{"text": system_prompt}],
        "messages": [
            {
                "role": "user",
                "content": [{"text": user_prompt}]
            }
        ],
        "inferenceConfig": {
            "maxTokens": max_tokens,
            "temperature": temperature,
            "topP": 0.9
        }
    }

    response = bedrock.invoke_model(
        modelId=BEDROCK_MODEL_ID,
        body=json.dumps(request_body),
        contentType="application/json",
        accept="application/json"
    )

    body = json.loads(response["body"].read())
    return body["output"]["message"]["content"][0]["text"].strip()


In [ ]:
PLAN_SYSTEM_PROMPT = load_prompt("planningPrompt.txt")

def generate_plan(df: pd.DataFrame):
    sample = df.head(5).to_dict(orient="records")

    user_prompt = (
        "You are designing a plan for analyzing this transaction dataset.\n\n"
        "Return ONLY a valid JSON object.\n\n"
        f"Sample rows:\n{json.dumps(sample, indent=2)}\n"
    )

    result = invoke_bedrock_json(PLAN_SYSTEM_PROMPT, user_prompt)
    (OUTPUT_DIR / "plan.json").write_text(json.dumps(result, indent=2))
    return result

df = pd.read_csv(TRANSACTION_FILE)
plan = generate_plan(df)
plan


{'plan': [{'step': 1,
   'goal': 'Understanding the dataset',
   'action': 'Load and inspect the dataset to identify fields, data types, and any missing or inconsistent data'},
  {'step': 2,
   'goal': 'Categorizing transactions',
   'action': 'Assign predefined categories to each transaction based on merchant and description'},
  {'step': 3,
   'goal': 'Computing KPIs',
   'action': 'Calculate key performance indicators such as total spending, average transaction amount, and category-wise spending'},
  {'step': 4,
   'goal': 'Summarizing results',
   'action': 'Generate a summary report with totals, averages, and categorized spending for the month'},
  {'step': 5,
   'goal': 'Reflecting on quality and improvements',
   'action': 'Review the dataset and analysis for any anomalies, missing data, or areas for improvement in categorization and reporting'}]}

In [177]:
CATEGORIZE_SYSTEM_PROMPT = load_prompt("categorizationPrompt.txt")

def chunk_list(items, size):
    return [items[i:i+size] for i in range(0, len(items), size)]


In [ ]:
def categorize_transactions(df: pd.DataFrame, batch_size=20):
    records = df.to_dict(orient="records")
    batches = chunk_list(records, batch_size)
    all_rows = []

    for idx, batch in enumerate(batches, start=1):
        user_prompt = (
            "You are a transaction categorization assistant.\n"
            "For each transaction, add a 'category' field.\n"
            "Return ONLY a JSON object like this:\n"
            '{ "categorized": [ { ... }, ... ] }\n\n'
            f"Transactions:\n{json.dumps(batch, indent=2)}"
        )

        result = invoke_bedrock_json(CATEGORIZE_SYSTEM_PROMPT, user_prompt)

        # Extract categorized rows safely
        if isinstance(result, dict) and "categorized" in result:
            rows = result["categorized"]
        elif isinstance(result, dict) and "raw_text" in result:
            try:
                parsed = json.loads(result["raw_text"])
                rows = parsed.get("categorized", [])
            except:
                print(f"Batch {idx}: raw model text not valid JSON\n")
                print(result["raw_text"])
                rows = []
        else:
            print(f"Batch {idx}: Unexpected format → {result}")
            rows = []

        all_rows.extend(rows)
        print(f"Batch {idx}/{len(batches)} complete. Added {len(rows)} rows.")

    output = {"categorized": all_rows}
    (OUTPUT_DIR / "categorized.json").write_text(json.dumps(output, indent=2))
    return output

categorized = categorize_transactions(df)
categorized["categorized"][:5]


Batch 1/2 complete. Added 20 rows.
Batch 2/2 complete. Added 14 rows.


[{'date': '2024-10-01',
  'merchant': 'Starbucks',
  'amount': 6.45,
  'category': 'Dining'},
 {'date': '2024-10-01',
  'merchant': 'Amazon',
  'amount': 32.89,
  'category': 'Shopping'},
 {'date': '2024-10-02',
  'merchant': 'Uber Eats',
  'amount': 24.1,
  'category': 'Dining'},
 {'date': '2024-10-03',
  'merchant': 'Comcast',
  'amount': 89.99,
  'category': 'Utilities'},
 {'date': '2024-10-03',
  'merchant': 'Payroll',
  'amount': -1800.0,
  'category': 'Income'}]

In [179]:
KPI_SYSTEM_PROMPT = load_prompt("kpiprompt.txt")

def compute_kpis_bedrock(categorized):
    rows = categorized["categorized"]
    user_prompt = (
        "Compute KPIs for the following categorized transactions:\n\n"
        f"{json.dumps(rows, indent=2)}"
    )
    result = invoke_bedrock_json(KPI_SYSTEM_PROMPT, user_prompt)
    (OUTPUT_DIR / "kpis.json").write_text(json.dumps(result, indent=2))
    return result

kpis = compute_kpis_bedrock(categorized)
kpis


{'kpis': {'total_spend': 1336.68,
  'total_income': 3600.0,
  'average_expense': 34.86,
  'top_merchants': [{'merchant': 'Amazon', 'count': 3},
   {'merchant': 'Payroll', 'count': 2},
   {'merchant': 'Target', 'count': 1}]}}

In [180]:
SUMMARY_SYSTEM_PROMPT = load_prompt("summaryPrompt.txt")

def generate_summary(kpis):
    user_prompt = f"Summarize these KPIs:\n\n{json.dumps(kpis, indent=2)}"
    text = invoke_bedrock_text(SUMMARY_SYSTEM_PROMPT, user_prompt)
    (OUTPUT_DIR / "summary.txt").write_text(text)
    return text

summary = generate_summary(kpis)
print(summary)

Your monthly income is $3,600, with total spending at $1,336.68. Average expense is $34.86. Top spenders include Amazon (3 times), Payroll (2 times), and Target (1 time). Overall, income comfortably covers spending.


In [182]:
REFLECTION_SYSTEM_PROMPT = load_prompt("reflectionPrompt.txt")

def generate_reflection(plan, categorized, kpis, summary):
    context = {
        "plan": plan,
        "categorized_sample": categorized["categorized"][:10],
        "kpis": kpis,
        "summary": summary
    }

    user_prompt = f"Here is the workflow output:\n\n{json.dumps(context, indent=2)}"
    text = invoke_bedrock_text(REFLECTION_SYSTEM_PROMPT, user_prompt)
    (OUTPUT_DIR / "reflection.txt").write_text(text)
    return text

reflection = generate_reflection(plan, categorized, kpis, summary)
print(reflection)


- **Potential Issues**
  - Inconsistent merchant categorization: "Target" is listed in top merchants but not in the sample data.
  - Missing data validation: No explicit check for missing or inconsistent data in the dataset.
  - Incomplete categorization: Some merchants might not have predefined categories.
  - KPI calculation inconsistencies: Potential issues with rounding or aggregation methods.
  - Summary report: Lack of detailed breakdown of spending by category.

- **Suggested Improvements**
  - Validate and standardize merchant categorization: Ensure all merchants are categorized correctly.
  - Implement data validation: Check for missing or inconsistent data before processing.
  - Expand predefined categories: Include more merchants in the categorization rules.
  - Review KPI calculation methods: Ensure consistency in rounding and aggregation.
  - Enhance summary report: Include detailed breakdowns of spending by category and merchant.

- **Next Iteration Focus**
  - Detailed v